# 02 · Modelos de Machine Learning — Predicción de generación renovable

**Pregunta:** ¿se puede predecir la generación mensual (GWh) de una central a partir de su fuente, ubicación, programa y estacionalidad? ¿Qué tan bien funciona según la región?

## Plan (decisiones tomadas en el EDA)
- **Objetivo:** `log1p(GWh)`. La generación es muy asimétrica; el log evita que unos pocos parques grandes dominen el entrenamiento. Las métricas finales se reportan en GWh.
- **Biodiésel excluido** (un único episodio en 2012, sin patrón aprendible).
- **Validación temporal, no aleatoria:** se entrena con lo anterior a 2025 y se testea en 2025–2026. Un split aleatorio mezclaría el futuro con el pasado.
- **Dos escenarios**, porque responden preguntas distintas:
  - **Estático (central nueva):** solo fuente, región, provincia, programa, mes y antigüedad.
  - **Con historial (central existente):** además, lo generado en los 3 meses previos.
- **Métrica principal: WAPE** = suma de errores absolutos / suma de lo real. Se interpreta como "error porcentual sobre el total", y no se distorsiona con meses de generación casi nula (a diferencia del MAPE).
- **Modelos del programa:** Regresión (Ridge) y Random Forest, con ajuste de hiperparámetros con **Optuna** (Clase 13), validación cruzada (Clase 16) y explicabilidad (Clase 17).

> El código reutilizable está en `src/` (preprocesamiento, modelos, evaluación); acá se usa y se interpreta.

In [ ]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance

from src import config
from src.preprocessing import preparar_datasets, split_temporal
from src.modelos import BaselineEstacional, pipeline_ridge, pipeline_rf, a_gwh
from src.evaluacion import metricas, error_por_grupo
from src.train import calcular_cuantiles_validacion

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
np.random.seed(config.SEMILLA)

## 1. Datos y partición temporal

In [ ]:
datasets = preparar_datasets()
for esc, d in datasets.items():
    tr, te = split_temporal(d)
    print(f"{esc:10s} → filas: {len(d):6,} | train (<{config.TEST_INICIO[:4]}): {len(tr):6,} | "
          f"test (≥{config.TEST_INICIO[:4]}): {len(te):5,}")
datasets["estatico"].head()

In [ ]:
d = datasets["estatico"]
tr, te = split_temporal(d)
nuevas = set(te["maquina"]) - set(tr["maquina"])
print(f"Máquinas en test: {te['maquina'].nunique()}  |  que NO existían en train: {len(nuevas)} "
      f"({len(nuevas) / te['maquina'].nunique():.0%})")

por_anio = d.groupby([d["mes"].dt.year, d["mes"] >= pd.Timestamp(config.TEST_INICIO)]).size().unstack(fill_value=0)
por_anio.columns = ["train", "test"]
por_anio.plot(kind="bar", stacked=True, color=["#3b7dd8", "#e07b39"], figsize=(10, 3.8))
plt.title("Filas por año: entrenamiento vs. test"); plt.xlabel(""); plt.show()

**Detalle importante:** ~17% de las máquinas del test **no existían** en el entrenamiento. El parque creció mucho (hallazgo 3 del EDA), así que el test es un examen realista de "¿generaliza a centrales que no vio?".

## 2. Escenario estático (central nueva)
### 2.1 Baselines y modelos con parámetros por defecto
Un modelo solo vale si le gana a algo simple. El baseline es el promedio histórico (en log) por **fuente y mes del año**.

In [ ]:
def evaluar_modelos(escenario: str, params_rf: dict | None = None) -> tuple[pd.DataFrame, dict]:
    """Entrena baselines y modelos con la partición temporal. Devuelve tabla de métricas y predicciones."""
    d = datasets[escenario]
    tr, te = split_temporal(d)
    cols = config.FEATURES[escenario]
    preds = {}

    preds["Baseline (media fuente×mes)"] = a_gwh(BaselineEstacional().fit(tr, tr["y_log"]).predict(te))
    if escenario == "historial":
        preds["Persistencia (mes anterior)"] = np.expm1(te["log_lag_1m"]).to_numpy()
    preds["Ridge"] = a_gwh(pipeline_ridge(escenario).fit(tr[cols], tr["y_log"]).predict(te[cols]))
    rf = pipeline_rf(escenario, params_rf).fit(tr[cols], tr["y_log"])
    nombre_rf = "Random Forest" + (" (Optuna)" if params_rf else " (defecto)")
    preds[nombre_rf] = a_gwh(rf.predict(te[cols]))

    tabla = pd.DataFrame({k: metricas(te["gwh"], v) for k, v in preds.items()}).T
    return tabla[["mae_gwh", "rmse_gwh", "wape", "mae_log", "r2_log"]], {"preds": preds, "rf": rf, "te": te, "tr": tr}

tabla_est, res_est = evaluar_modelos("estatico")
tabla_est

### 2.2 Validación cruzada: temporal vs. aleatoria (Clase 16)
Comparo dos esquemas con el mismo modelo:
- **Temporal (ventanas expansivas):** entrena con lo anterior a cada año (2022, 2023, 2024) y valida en ese año. Imita el uso real: predecir el futuro.
- **Aleatoria (KFold):** mezcla filas de todos los años. Es el error "de manual", pero con series temporales puede filtrar información del futuro.

Se suele esperar que la aleatoria sea más optimista. **¿Pasa acá?**

In [ ]:
def wape_fold(escenario, tr, va):
    cols = config.FEATURES[escenario]
    modelo = pipeline_rf(escenario).fit(tr[cols], tr["y_log"])
    return metricas(va["gwh"], a_gwh(modelo.predict(va[cols])))["wape"]

filas = []
for esc in ("estatico", "historial"):
    d_pre = datasets[esc][datasets[esc]["mes"] < pd.Timestamp(config.TEST_INICIO)].reset_index(drop=True)
    temporal = [wape_fold(esc, d_pre[d_pre["mes"] < f"{a}-01-01"],
                          d_pre[(d_pre["mes"] >= f"{a}-01-01") & (d_pre["mes"] < f"{a + 1}-01-01")])
                for a in config.ANIOS_VALIDACION_OPTUNA]
    aleatoria = [wape_fold(esc, d_pre.iloc[i], d_pre.iloc[j])
                 for i, j in KFold(5, shuffle=True, random_state=config.SEMILLA).split(d_pre)]
    filas.append({"escenario": esc, "WAPE temporal": np.mean(temporal), "± temporal": np.std(temporal),
                  "WAPE aleatoria": np.mean(aleatoria), "± aleatoria": np.std(aleatoria)})
pd.DataFrame(filas).set_index("escenario")

**Resultado y cómo interpretarlo:** en este dataset la CV aleatoria **no** salió más optimista; en el escenario con historial incluso dio un error mayor. No es una contradicción: los dos esquemas **evalúan períodos distintos** (la aleatoria incluye los años tempranos 2011-2018, con pocas centrales chicas y más difíciles de predecir; la temporal solo valida en 2022-2024), así que los números no son directamente comparables.
Lo que sí se mantiene es el argumento metodológico: la validación temporal responde la pregunta que importa ("¿qué tan bien predigo meses que todavía no vi?") y evita el riesgo de fuga de información cuando se usan rezagos. Por eso se usa en Optuna y en el test final.

### 2.3 Ajuste de hiperparámetros con Optuna (Clase 13)
Optuna busca combinaciones de hiperparámetros de forma inteligente (muestreo bayesiano TPE) en lugar de probar todas como `GridSearchCV`.
- **Qué se optimiza:** el error absoluto medio en escala log, promediado en 3 ventanas temporales (validando en 2022, 2023 y 2024).
- **Qué NO se toca:** el test (2025-2026) queda intacto hasta la evaluación final.
- Acoté `max_depth` a propósito: árboles muy profundos pesaban más de 100 MB sin mejorar el error, y el modelo debe poder desplegarse en una API.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 25   # bajalo a 10 si tu equipo es lento

def crear_objective(escenario: str):
    d = datasets[escenario]
    d = d[d["mes"] < pd.Timestamp(config.TEST_INICIO)]          # el test nunca entra acá
    cols = config.FEATURES[escenario]

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 250, step=50),
            "max_depth": trial.suggest_int("max_depth", 6, 14),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 20),
            "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        }
        errores = []
        for anio in config.ANIOS_VALIDACION_OPTUNA:
            tr_f = d[d["mes"] < f"{anio}-01-01"]
            va_f = d[(d["mes"] >= f"{anio}-01-01") & (d["mes"] < f"{anio + 1}-01-01")]
            modelo = pipeline_rf(escenario, params).fit(tr_f[cols], tr_f["y_log"])
            errores.append(np.mean(np.abs(va_f["y_log"] - modelo.predict(va_f[cols]))))
        return float(np.mean(errores))
    return objective

estudios = {}
for esc in ("estatico", "historial"):
    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=config.SEMILLA))
    estudio.optimize(crear_objective(esc), n_trials=N_TRIALS)
    estudios[esc] = estudio
    print(f"{esc:10s} mejor MAE(log) en validación: {estudio.best_value:.4f} | {estudio.best_params}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for a, (esc, est) in zip(ax, estudios.items()):
    vals = [t.value for t in est.trials]
    a.plot(vals, "o", alpha=0.5, label="cada prueba")
    a.plot(np.minimum.accumulate(vals), lw=2, label="mejor hasta ahora")
    a.set(title=f"Optuna — escenario {esc}", xlabel="prueba", ylabel="MAE (log) validación"); a.legend()
plt.tight_layout(); plt.show()

# Guardar para que `python -m src.train` use estos hiperparámetros
config.MODELOS_DIR.mkdir(exist_ok=True)
mejores = {esc: est.best_params for esc, est in estudios.items()}
config.RUTA_MEJORES_PARAMS.write_text(json.dumps(mejores, indent=2), encoding="utf-8")
print("Guardado:", config.RUTA_MEJORES_PARAMS)

In [ ]:
tabla_est, res_est = evaluar_modelos("estatico", mejores["estatico"])
tabla_est

## 3. Escenario con historial (central existente)
Ahora el modelo conoce los 3 meses previos de la central. Se compara contra una **persistencia** ingenua: "este mes generará lo mismo que el mes pasado". Si el modelo no le gana a eso, el aprendizaje automático no está aportando valor.

In [ ]:
tabla_his, res_his = evaluar_modelos("historial", mejores["historial"])
tabla_his

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharex=False)
for a, (titulo, tabla) in zip(ax, [("Central nueva (estático)", tabla_est), ("Con historial", tabla_his)]):
    tabla["wape"].sort_values(ascending=False).plot.barh(ax=a, color="#3b7dd8")
    a.set(title=titulo, xlabel="WAPE en test (menor = mejor)")
    for i, v in enumerate(tabla["wape"].sort_values(ascending=False)):
        a.text(v + 0.01, i, f"{v:.0%}", va="center")
    a.set_xlim(0, max(tabla["wape"]) * 1.2)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (titulo, res) in zip(ax, [("Estático", res_est), ("Con historial", res_his)]):
    nombre = [k for k in res["preds"] if k.startswith("Random")][0]
    a.scatter(np.log1p(res["te"]["gwh"]), np.log1p(res["preds"][nombre]), s=6, alpha=0.25)
    lim = [0, np.log1p(res["te"]["gwh"]).max()]
    a.plot(lim, lim, "r--", lw=1)
    a.set(title=f"{titulo}: real vs. predicho (log1p GWh)", xlabel="real", ylabel="predicho")
plt.tight_layout(); plt.show()

## 4. Error por subgrupo (sesgos y equidad del modelo)
Un error global bueno puede esconder que el modelo falla en ciertos grupos. Acá se mide el error **por fuente y por región**, junto con la cantidad de máquinas (grupos chicos dan métricas inestables) y el **sesgo** (error medio con signo: negativo = el modelo subestima).

In [ ]:
te_est = res_est["te"].assign(pred=res_est["preds"]["Random Forest (Optuna)"])
te_his = res_his["te"].assign(pred=res_his["preds"]["Random Forest (Optuna)"])

print("ESTÁTICO — por fuente");   display(error_por_grupo(te_est, "fuente"))
print("ESTÁTICO — por región");   display(error_por_grupo(te_est, "region"))

In [ ]:
print("CON HISTORIAL — por fuente"); display(error_por_grupo(te_his, "fuente"))
print("CON HISTORIAL — por región"); display(error_por_grupo(te_his, "region"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for a, (titulo, df_) in zip(ax, [("Estático", te_est), ("Con historial", te_his)]):
    g = error_por_grupo(df_, "region").sort_values("wape")
    barras = a.barh(g["region"], g["wape"], color="#e07b39")
    for b, n in zip(barras, g["maquinas"]):
        a.text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2, f"{n} máq.", va="center", fontsize=9)
    a.set(title=f"{titulo}: WAPE por región", xlabel="WAPE")
    a.set_xlim(0, g["wape"].max() * 1.25)
plt.tight_layout(); plt.show()

### Centrales conocidas vs. centrales nuevas
¿Le va peor el modelo estático con máquinas que **nunca vio** en el entrenamiento?

In [ ]:
vistas = set(res_est["tr"]["maquina"])
te_est["grupo"] = np.where(te_est["maquina"].isin(vistas), "máquina vista en train", "máquina NUEVA")
display(error_por_grupo(te_est, "grupo"))

## 5. Rango de incertidumbre: ¿está calibrado?
La API devuelve un rango (percentiles 10–90 del error). Si está bien calibrado, ~80% de los valores reales deberían caer dentro. Los percentiles se calcularon con un año de validación (2024) **anterior** al test, así que esta verificación es honesta.

In [ ]:
def cobertura(escenario: str, res: dict, params: dict) -> pd.DataFrame:
    q = calcular_cuantiles_validacion(datasets[escenario], escenario, params)
    te = res["te"].copy()
    cols = config.FEATURES[escenario]
    pred_log = res["rf"].predict(te[cols])
    lo = np.array([q.get(f, q["_global"])[0] for f in te["fuente"]])
    hi = np.array([q.get(f, q["_global"])[1] for f in te["fuente"]])
    te["dentro"] = (te["y_log"] >= pred_log + lo) & (te["y_log"] <= pred_log + hi)
    return te.groupby("fuente")["dentro"].mean().rename(escenario).to_frame().T.assign(
        global_=te["dentro"].mean())

cob = pd.concat([cobertura("estatico", res_est, mejores["estatico"]),
                 cobertura("historial", res_his, mejores["historial"])])
display((cob * 100).round(0))

## 6. Interpretabilidad: ¿qué usa el modelo? (Clase 17)
### 6.1 Importancia por permutación (scikit-learn)
Se mezcla al azar una variable en el test y se mide cuánto empeora el error. Si empeora mucho, el modelo la usa mucho.

In [ ]:
def importancias(escenario: str, res: dict) -> pd.Series:
    te, cols = res["te"], config.FEATURES[escenario]
    r = permutation_importance(res["rf"], te[cols], te["y_log"], n_repeats=5,
                               scoring="neg_mean_absolute_error", random_state=config.SEMILLA)
    return pd.Series(r.importances_mean, index=cols).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, (esc, res) in zip(ax, [("estatico", res_est), ("historial", res_his)]):
    imp = importancias(esc, res)
    imp.plot.barh(ax=a, color="#3b7dd8")
    a.set(title=f"Importancia por permutación — {esc}", xlabel="aumento del MAE (log) al mezclar la variable")
plt.tight_layout(); plt.show()

### 6.2 SHAP (opcional)
SHAP reparte la predicción de **cada fila** entre las variables. El programa lo ve de forma conceptual; esta celda es una primera aproximación práctica (`pip install shap`).

In [ ]:
try:
    import shap
    modelo = res_est["rf"]
    muestra = res_est["te"][config.FEATURES["estatico"]].sample(300, random_state=config.SEMILLA)
    X_t = modelo.named_steps["prep"].transform(muestra)
    nombres = modelo.named_steps["prep"].get_feature_names_out()
    valores = shap.TreeExplainer(modelo.named_steps["modelo"]).shap_values(X_t)
    shap.summary_plot(valores, X_t, feature_names=nombres, max_display=12)
except ImportError:
    print("SHAP no está instalado (pip install shap). Celda opcional.")

**Cómo leerlo (y cómo no):** una variable importante significa que el modelo la **usa**, no que **cause** la generación. Además, región y fuente están casi confundidas (por ejemplo, Patagonia es ~96% eólica), así que el modelo no puede separar bien el efecto de una del de la otra.

## 7. Conclusiones
Los números exactos dependen de tu corrida de Optuna; lo que sigue vale para los valores observados (ver tablas de arriba) y conviene que **actualices las cifras con las tuyas**.

**1. Central nueva (escenario estático).** El Random Forest baja el error de ~71 % (baseline fuente×mes) a ~47-52 % WAPE. Es una mejora real, pero el error sigue siendo alto: sin potencia instalada ni clima, el modelo no distingue un parque chico de uno grande de la misma fuente y provincia. La importancia por permutación lo confirma: pesan casi solo `fuente` (≈0,45), `programa` (≈0,19), `región` y `provincia`; la estacionalidad aporta poco.

**2. Central existente (con historial).** La persistencia ("lo mismo que el mes pasado") ya logra ~15 % WAPE y el Random Forest ~12-13 %: una mejora de alrededor del 15 % sobre un método trivial. Casi todo el valor viene de los datos del mes anterior (`log_lag_1m` ≈0,86 de importancia; el resto de las variables, cerca de 0), no del modelo en sí. *Reportar solo "R² = 0,97" sería engañoso.*

**3. Las centrales nuevas son el punto débil.** En el modelo estático, el WAPE sube de ~49 % (máquinas ya vistas en el entrenamiento) a ~73 % (máquinas nuevas), con R² ≈0,15. Justo el caso de uso más interesante (¿cuánto generará un parque nuevo?) es donde peor funciona.

**4. Subestimación sistemática.** El sesgo es negativo en eólica y en Buenos Aires/Patagonia: el modelo predice menos de lo que ocurre. *Hipótesis (no verificable con estos datos):* los parques nuevos de 2025-26 son más grandes que los del pasado y, sin potencia instalada, el modelo no puede saberlo.

**5. Desempeño desigual por región.** En el modelo estático el WAPE va de ~19 % (Comahue) y ~32 % (Noroeste) a ~89 % (Cuyo, R² ≈0,1). Con historial el error se parece más entre regiones (~10-21 %). Litoral (6 máquinas) y Gran Bs.As. (8) tienen pocos datos, así que sus métricas son inestables.

**6. Rango de incertidumbre.** El rango 10-90 % de la API cubre ~78-79 % de los casos reales en test (objetivo: 80 %), con menor cobertura en solar (~72-75 %). Está razonablemente calibrado, pero hay que presentarlo como aproximado.

**7. Validación.** La CV temporal vs. la aleatoria no mostró la diferencia esperada (ver 2.2); se mantiene la temporal por ser la que imita el uso real.

### Qué se lleva a la API
- `POST /predict` → modelo **estático** (central nueva). Devuelve un rango y avisos cuando la combinación fuente/región/provincia es inédita.
- `POST /predict/historial` → modelo **con historial** (los 3 meses previos).
- Los hiperparámetros se guardan en `models/best_params.json` y `python -m src.train` entrena y serializa ambos modelos.

### Para el README (sesgos y uso responsable)
- El modelo **no debe usarse** para decidir inversiones ni despacho: no conoce potencia ni clima, y su error en centrales nuevas es alto.
- Sesgo de supervivencia: solo hay centrales que se construyeron.
- Región y fuente están casi confundidas: no interpretar la importancia de variables como causalidad.
- El parque cambió mucho en pocos años; el modelo puede degradarse con tecnologías o regulaciones nuevas.